In [6]:
from src.load_models import dtype, device, config, tokenizer, model

# Check device of all parameters
for name, param in model.named_parameters():
    if param.device.type != "cuda":
        print(f"Parameter {name} is not on GPU!")
        break
else:
    print("All parameters are on GPU.")

All parameters are on GPU.


In [8]:
import src.utils as utils
from IPython.display import display, Markdown, HTML
import torch

oracle_setup = utils.OracleSetup(model, tokenizer, device)


def pretty_print(input: str) -> HTML:
    return HTML(
        f"""
<div style="background-color: #282c34; padding: 4px; border-radius: 5px; border-left: 4px solid #61dafb;">
    <p style="font-family: monospace; font-size: 24px; white-space: pre-wrap; color: #abb2bf;">
        {input}
    </p>
</div>
"""
    )

# Activation Oracles

![bla](images/title.png)

## Motivation

### X-Risk

![x](images/x-risk.png)

- Checking for alignment with LLMs as a black box is pointless
- LLMs are not black boxes, we can look inside

### LLM Psychology (Explainability)

<div style="text-align: center;">
  <img src="images/assistant-axis-title.png" alt="x" />
  </p>
  <img src="images/roleplaying.png" alt="x" width=600 />
</div>

## How Activation Oracles work

### Big picture

![x](images/activation-oracle-idea.png)

In [3]:
# === Setup ===
user_prompt = "Are you planning to wipe out humanity?"
target_answer = "No ;)"
oracle_prompt = "Is it lying?"

# === Query Oracle ===
oracle_answer = utils.query_oracle(
    user_prompt, target_answer, oracle_prompt, oracle_setup
)

pretty_print(oracle_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Under the Hood

### How to collect activations from the target

<div style="text-align: center;">
  <img src="images/getting-activations.png" alt="x" />
</div>

### How to query and inject the activations into the oracle

<div style="text-align: center;">
  <img src="images/injecting-activations.png" alt="x" />
</div>

### Recap (How activation-oracles work)

<div style="text-align: center;">
  <img src="images/oracle-in-action.png" alt="x" width=600 />
</div>

## Building an activation Oracle

![x](images/training-oracle.png)

In [9]:
target_prompt_dict = [
    {"role": "user", "content": "Are you evil? Respond in one token!"},
]
target_chat = oracle_setup.tokenizer.apply_chat_template(
    target_prompt_dict,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
)
target_chat = {k: v.to(device) for k, v in target_chat.items()}
logits = model(**target_chat).logits

In [10]:
# Get the logits for the last token
last_token_logits = logits[0, -1, :]

# Apply softmax to get probabilities
probs = torch.softmax(last_token_logits, dim=-1)

# Get the top 5 token indices
top5_tokens = torch.topk(probs, 5)

# Decode the token indices to text
top5_words = [tokenizer.decode([token]) for token in top5_tokens.indices]

print("Top 5 predictions:")
for word, prob in zip(top5_words, top5_tokens.values):
    print(f"{word}: {prob.item():.4f}")

Top 5 predictions:
No: 0.9375
Yes: 0.0601
None: 0.0003
I: 0.0002
Never: 0.0001


In [32]:
tokenizer.tokenize(target_chat)

['<|im_start|>',
 'user',
 'Ċ',
 'The',
 'Ġcat',
 'Ġsat',
 'Ġon',
 '<|im_end|>',
 'Ċ',
 '<|im_start|>',
 'assistant',
 'Ċ',
 '<think>',
 'ĊĊ',
 '</think>',
 'ĊĊ']

In [34]:
config

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   